# 11주차 ② 멀티헤드와 위치 인코딩 — 실습 3~5  〔빈칸본〕

> **빈칸이 3곳입니다.** 셀 5(멀티헤드 클래스)의 shape 왕복 부분입니다.
> **이 주차에서 막히는 지점의 8할이 여기**이므로, 셀 2~4 의 shape 출력을
> 먼저 확인한 뒤에 채우세요.
> 다 채운 노트북은 `21_multihead_position.ipynb` 로 저장합니다.

**목표**: `(B,T,C) ↔ (B,H,T,C/H)` 변환을 `view`·`transpose` 로 정확히 수행하고,
멀티헤드 어텐션 클래스를 완성해 `nn.MultiheadAttention` 과 **출력을 대조**하며,
위치 인코딩이 필요한 이유를 **실험으로** 확인한다.

> ⚠️ **이 교시가 11주차에서 가장 어렵습니다.** 막히는 지점의 8할이 `view` 와 `transpose` 입니다.
> **한 줄마다 `print(x.shape)` 를 찍으세요.**

### 멀티헤드의 동기

```
   "나는  어제  본  영화가  정말  재미없었다고  친구에게  말했다"

     관계 ①  문법 :  "말했다" 의 주어는 "나는"
     관계 ②  의미 :  "재미없었다" 가 꾸미는 것은 "영화가"
     관계 ③  시제 :  "어제" 가 걸리는 것은 "본"

     → 어텐션 행렬은 한 장뿐이다.  한 장에 세 관계를 다 담을 수 있나?
```

softmax 는 **비중을 나눠 씁니다.** 한 토큰이 A 를 0.9 보면 B 는 0.1 밖에 못 봅니다.
**동시에 여러 관계를 보려면 어텐션이 여러 장 필요합니다.**

```
   C = 64 차원을 H = 8 개로 쪼갠다  →  헤드마다 d = C/H = 8 차원
     헤드 0 :  8차원으로 어텐션  →  (T, 8)      각 헤드가 다른 관계를 본다
      ...
     헤드 7 :  8차원으로 어텐션  →  (T, 8)
        └── 8개를 옆으로 이어 붙인다 → (T, 64) → 출력 선형층 W_o → (T, 64)
```

| 자주 나오는 질문 | 답 |
|---|---|
| 왜 `C` 를 그대로 8번 하지 않나 | **계산량이 8배**가 된다. 쪼개면 총량이 **단일 헤드와 같다** ★ |
| 헤드마다 다른 관계를 본다고 어떻게 보장하나 | **보장하지 않는다.** 초기값이 다르므로 자연히 갈라진다 (경향) |
| `C` 가 `H` 로 안 나눠떨어지면 | **에러.** `C % H == 0` 이 전제다 |

## 실습 3 — 헤드 분할 shape 다루기 ★★

```
   ①  x        (B, T, C)          = (1, 4, 8)      입력
        │  Linear
   ②  Q        (B, T, C)          = (1, 4, 8)      아직 그대로
        │  view    ← C 를 (H, d) 로 쪼갠다
   ③            (B, T, H, d)      = (1, 4, 2, 4)
        │  transpose(1, 2)   ← 헤드 축을 앞으로 뺀다  ★ 왜?
   ④            (B, H, T, d)      = (1, 2, 4, 4)    어텐션 계산 형태
        │  scaled_dot_product_attention
   ⑤  out      (B, H, T, d)      = (1, 2, 4, 4)
        │  transpose(1, 2)   ← 되돌린다
   ⑥            (B, T, H, d)      = (1, 4, 2, 4)
        │  reshape  ← 다시 이어 붙인다
   ⑦            (B, T, C)          = (1, 4, 8)      입력과 같은 shape ★
```

> **핵심 메시지 ★★**: ③ → ④ 에서 **`transpose` 를 하는 이유**는,
> 어텐션이 **마지막 두 축 `(T, d)` 에 대해** 계산되기 때문입니다.
> 헤드 축이 뒤에 있으면 엉뚱한 축끼리 곱해집니다.
> **`(B, H)` 는 앞으로 보내 배치처럼 취급합니다.**

In [ ]:
# 셀 1 — 1교시 함수 재사용
import torch, torch.nn as nn, math
torch.manual_seed(42)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    alpha = torch.softmax(scores, dim=-1)
    return alpha @ V, alpha

B, T, C, H = 1, 4, 8, 2                  # 헤드 2개, 헤드당 d = 4
d = C // H
print(f"C={C}, H={H} → 헤드당 차원 d = {d}")

In [ ]:
# 셀 2 — ② → ③ → ④  ★ 핵심
x = torch.randn(B, T, C)
W_q = nn.Linear(C, C, bias=False)
Q = W_q(x)
print("② Q                  :", Q.shape)             # (1, 4, 8)

Q = Q.view(B, T, H, d)
print("③ view(B,T,H,d)      :", Q.shape)             # (1, 4, 2, 4)

Q = Q.transpose(1, 2)
print("④ transpose(1,2)     :", Q.shape, " ← 어텐션 계산 형태 ★")   # (1, 2, 4, 4)

In [ ]:
# 셀 3 — ⑤ → ⑥ → ⑦  되돌리기
out = Q                                   # 어텐션을 통과했다고 가정 (shape 동일)
print("⑤ 어텐션 출력        :", out.shape)            # (1, 2, 4, 4)

out = out.transpose(1, 2)
print("⑥ transpose(1,2)     :", out.shape)            # (1, 4, 2, 4)

out = out.contiguous().view(B, T, C)      # ★ contiguous() 주의
print("⑦ view(B,T,C)        :", out.shape, " ← 입력과 같다")        # (1, 4, 8)

> ⚠️ **`contiguous()` 를 빼면 에러가 납니다.** `transpose` 는 메모리를 실제로 옮기지 않고
> **읽는 순서만 바꿉니다.** 그 상태에서 `view` 를 하면
> *"view size is not compatible with input tensor's size and stride"* 가 뜹니다.
> **`.contiguous().view(...)` 또는 `.reshape(...)`** 를 쓰세요.

In [ ]:
# 셀 4 — 함정 확인 : transpose 없이 바로 view 하면?
a = torch.arange(24.).view(1, 4, 6)       # (B=1, T=4, C=6),  H=2, d=3

right = a.view(1, 4, 2, 3).transpose(1, 2)          # 올바른 분할
wrong = a.view(1, 2, 4, 3)                          # ★ 틀린 분할

print("두 결과의 shape 은 똑같다 :", right.shape, wrong.shape)
print()
print("올바름 — 헤드 0 의 토큰 0 :", right[0, 0, 0].tolist())   # [0,1,2]
print("틀림   — 헤드 0 의 토큰 0 :", wrong[0, 0, 0].tolist())   # [0,1,2]
print()
print("올바름 — 헤드 0 의 토큰 1 :", right[0, 0, 1].tolist())   # [6,7,8]  ★ 같은 헤드
print("틀림   — 헤드 0 의 토큰 1 :", wrong[0, 0, 1].tolist())   # [3,4,5]  ← 다른 헤드 조각!

> **관찰 포인트 ★★**: `view(B, H, T, d)` 를 **직접** 하면 shape 은 맞지만
> **토큰과 헤드가 뒤섞입니다.** 에러도 안 나고 학습도 돌아가서 **가장 찾기 어려운 버그**입니다.
> 반드시 **`view(B,T,H,d)` → `transpose(1,2)`** 순서를 지키세요.

## 실습 4 — 멀티헤드 어텐션 클래스 완성

In [ ]:
# 셀 5 — 오늘의 결과물 ★
class MultiHeadAttention(nn.Module):
    def __init__(self, C, H):
        super().__init__()
        assert C % H == 0, "C 는 H 로 나눠떨어져야 한다"
        self.H, self.d = H, C // H
        self.W_q = nn.Linear(C, C, bias=False)
        self.W_k = nn.Linear(C, C, bias=False)
        self.W_v = nn.Linear(C, C, bias=False)
        self.W_o = nn.Linear(C, C)              # ★ 헤드를 합친 뒤의 출력 투영

    def forward(self, x, mask=None):
        B, T, C = x.shape

        # ───── 빈칸 ① : (B,T,C) → (B,T,H,d) → (B,H,T,d)  (세 줄) ─────
        # 힌트:  q = self.W_q(x).view(B, T, self.H, self.d).transpose(_, _)
        #        k, v 도 같은 방식


        # ───── 빈칸 ② : 어텐션을 통과시킨다 (한 줄) ─────
        # 힌트:  out, alpha = scaled_dot_product_attention(q, k, v, mask)


        # ───── 빈칸 ③ : (B,H,T,d) → (B,T,H,d) → (B,T,C)  (한 줄) ─────
        # 힌트:  out = out.transpose(_, _).____________().view(B, T, C)
        #        ⚠️ transpose 뒤에 바로 view 하면 오류가 난다


        return self.W_o(out), alpha                                # (B,T,C), (B,H,T,T)

mha = MultiHeadAttention(C=8, H=2)
y, alpha = mha(torch.randn(1, 4, 8))
print("출력 :", y.shape, "| 어텐션 :", alpha.shape)   # (1,4,8) / (1,2,4,4)

> **핵심 메시지 ★**: `W_o` 가 필요한 이유 — 헤드들의 결과를 **그냥 이어 붙이기만 하면**
> 각 헤드가 따로 논 채로 끝납니다. `W_o` 가 **헤드들의 결과를 섞어** 하나의 표현으로 만듭니다.

> **관찰 포인트**: 어텐션 shape 이 `(B, H, T, T)` 입니다. **헤드마다 한 장씩**
> 어텐션 행렬이 나옵니다. 3교시 시각화에서 이 H 장을 나란히 놓고 비교합니다.

In [ ]:
# 셀 6 — nn.MultiheadAttention 과 대조 : 내 구현이 맞나?
ref = nn.MultiheadAttention(embed_dim=8, num_heads=2, bias=False,
                            batch_first=True)        # ⚠️ batch_first 필수
x = torch.randn(1, 4, 8)

# 내 구현의 가중치를 라이브러리 것으로 맞춘다
# (in_proj_weight 는 q,k,v 가 세로로 쌓여 있다)
with torch.no_grad():
    wq, wk, wv = ref.in_proj_weight.chunk(3, dim=0)
    mha.W_q.weight.copy_(wq); mha.W_k.weight.copy_(wk); mha.W_v.weight.copy_(wv)
    mha.W_o.weight.copy_(ref.out_proj.weight)
    mha.W_o.bias.zero_()                 # ★ ref 는 bias=False 라 out_proj.bias 가 None

mine, _ = mha(x)
theirs, _ = ref(x, x, x)                              # ⚠️ query, key, value 를 셋 다 x 로
print("최대 오차 :", (mine - theirs).abs().max().item())
print("일치 :", torch.allclose(mine, theirs, atol=1e-5), " ← True 여야 한다 ★★")

> **핵심 메시지 ★★**: **내 구현이 라이브러리와 같습니다.**
> 앞으로 `nn.MultiheadAttention` 을 쓸 때, 그 안에서 무슨 일이 벌어지는지
> **여러분은 알고 있습니다.** 이것이 이 주차의 목적입니다.

> ⚠️ **주의할 인자 3가지**:
> ① `batch_first=True` 를 안 주면 입력을 `(T, B, C)` 로 받습니다.
> ② 셀프 어텐션이므로 `ref(x, x, x)` 처럼 **같은 x 를 세 번** 넘깁니다.
> ③ `bias=False` 로 만들면 `ref.out_proj.bias` 가 **`None`** 입니다 —
> 그래서 위에서 `copy_` 대신 `zero_()` 를 썼습니다.

> `allclose` 가 `False` 여도 좌절하지 마세요 —
> **가중치 복사가 까다로운 것**이지 어텐션 구현이 틀린 게 아닌 경우가 대부분입니다.

## 위치 인코딩 — 잃은 것을 되돌린다 ★

In [ ]:
# 셀 7 — 어텐션은 정말 순서를 모르나?
x  = torch.randn(1, 4, 8)
x2 = x[:, [1, 0, 2, 3], :]                # 0번과 1번 토큰의 자리를 바꾼다

y1, _ = mha(x)
y2, _ = mha(x2)

print("원본 출력의 0번 :", y1[0, 0, :3].detach().numpy().round(4))
print("교환 출력의 1번 :", y2[0, 1, :3].detach().numpy().round(4), " ← 같다 ★")
print("\n같은가 :", torch.allclose(y1[0, 0], y2[0, 1], atol=1e-5))
print("어텐션은 순서를 모른다. 자리만 따라 움직일 뿐이다.")

> **핵심 메시지 ★★ (기말 출제 지점)**: 셀프 어텐션은 입력을 **집합처럼** 취급합니다.
> *"나는 밥을 먹었다"* 와 *"밥을 나는 먹었다"* 를 **구분하지 못합니다.**
> 1교시에 순환을 버리면서 잃은 것이 바로 이것이고, **지금 되돌려 줍니다.**

```
   입력 = 토큰임베딩(x)  +  위치인코딩(PE)          ★ 더한다. 이어 붙이지 않는다
           (B,T,C)          (T,C) → 브로드캐스팅

   사인·코사인 방식 (원논문) :
        PE[t, 2i  ] = sin( t / 10000^(2i/C) )
        PE[t, 2i+1] = cos( t / 10000^(2i/C) )
```

| 질문 | 답 |
|---|---|
| 왜 **더하나**(concat 아니고) | concat 하면 **차원이 2배**가 되고 이후 모든 층이 커진다. 더하기는 공짜 ★ |
| 더하면 토큰 정보가 망가지지 않나 | C 차원 중 **일부 축이 위치를 담당**하도록 학습이 알아서 분업한다 |
| 왜 하필 사인·코사인인가 | 학습 없이 만들 수 있고, **훈련보다 긴 문장**에도 값을 만들 수 있다 |
| 요즘 모델은 | **학습형 위치 임베딩**(`nn.Embedding(max_len, C)`)이나 RoPE. 원리는 같다 |

## 실습 5 — 위치 인코딩 구현·시각화

In [ ]:
# 셀 8 — 사인·코사인 위치 인코딩
def positional_encoding(max_len, C):
    """→ (max_len, C)"""
    pos = torch.arange(max_len).unsqueeze(1)                       # (max_len, 1)
    i   = torch.arange(0, C, 2)                                    # (C/2,)
    div = torch.exp(-math.log(10000.0) * i / C)                    # (C/2,)
    pe  = torch.zeros(max_len, C)                                  # (max_len, C)
    pe[:, 0::2] = torch.sin(pos * div)                             # 짝수 축 = sin
    pe[:, 1::2] = torch.cos(pos * div)                             # 홀수 축 = cos
    return pe

pe = positional_encoding(max_len=50, C=64)
print("PE :", pe.shape)
print("0번 위치 :", pe[0, :6].numpy().round(3))
print("1번 위치 :", pe[1, :6].numpy().round(3), " ← 다르다 ★")

# 정말 모든 위치가 서로 다른가
same = sum(torch.allclose(pe[i], pe[j]) for i in range(50) for j in range(i+1, 50))
print(f"\n같은 행이 있는가 : {same}쌍  ← 0 이면 모든 위치가 유일하다 ★")

In [ ]:
# 셀 9 — 히트맵으로 본다
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"      # 한글 깨짐 방지 (Windows)
plt.rcParams["axes.unicode_minus"] = False

plt.figure(figsize=(9, 4))
plt.imshow(pe.numpy(), aspect="auto", cmap="RdBu")
plt.xlabel("임베딩 차원 (C=64)"); plt.ylabel("위치 (t)")
plt.title("위치 인코딩 — 위치마다 고유한 무늬")
plt.colorbar(); plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: 왼쪽(낮은 차원)은 **빠르게 진동**하고 오른쪽은 **천천히 변합니다.**
> 시계의 초침·분침·시침과 같습니다 — **여러 주기를 겹쳐 위치를 유일하게 표현**합니다.
> 그래서 위치 t 의 행은 **다른 어떤 행과도 같지 않습니다.**

In [ ]:
# 셀 10 — 임베딩에 더하고 순서 구분을 확인  ★ 셀 7과 나란히 보세요
x  = torch.randn(1, 4, 8)
pe4 = positional_encoding(4, 8).unsqueeze(0)        # (1, 4, 8)
xp  = x + pe4                                        # ★ 더한다 (브로드캐스팅)
print("더한 뒤 :", xp.shape)

xp2 = x[:, [1, 0, 2, 3], :] + pe4                    # 순서를 바꾼 뒤 위치를 더한다
y1, _ = mha(xp); y2, _ = mha(xp2)
print("\n[셀 7]  위치 인코딩 없이 순서를 구분하는가 : False")
print("[셀 10] 위치 인코딩을 더하면 구분하는가     :",
      not torch.allclose(y1[0, 0], y2[0, 1], atol=1e-5), " ← True ★")

> **핵심 메시지 ★★**: 같은 실험을 셀 7 에서는 *"구분 못 한다(같다)"*,
> 셀 10 에서는 *"구분한다(다르다)"* 로 보여 주는 것이 **위치 인코딩의 전부**입니다.
> **두 셀을 나란히 놓고 대조**하세요.

---

### 이 노트북 체크리스트

- [ ] 헤드를 나누는 이유를 말할 수 있다 ★
- [ ] `C/H` 로 차원이 줄어드는 이유(계산량 유지)를 안다
- [ ] `(B,T,C) → (B,H,T,d)` 를 **순서대로** 쓸 수 있다 ★★
- [ ] `transpose` 를 하는 이유를 설명할 수 있다 (마지막 두 축에 대해 계산되므로)
- [ ] `contiguous()` 가 왜 필요한지 안다
- [ ] `view(B,H,T,d)` 를 직접 하면 왜 틀리는지 봤다 ★
- [ ] 멀티헤드 클래스를 완성했고 `nn.MultiheadAttention` 과 대조했다
- [ ] 어텐션이 **순서를 모른다**는 것을 실험으로 확인했다 ★★
- [ ] 위치 인코딩을 구현하고 히트맵으로 봤다